<div style='text-align:right;font-size:0.9em'><a href='B8_Calculator_and_summary.ipynb'>← B8 Design calculator and summary</a> &nbsp;·&nbsp; <a href='00_Start_here.ipynb'>Contents</a></div>

# C – Your own corrosion-control design

<div style='background:rgba(31,119,180,0.10);border-left:4px solid #1f77b4;padding:8px 14px'>

**With this worksheet you can …**
- set up the design basis for the corrosion control of your semester-project plant, with a source for every value
- see how your pre-treatment (coagulation, membranes, ion exchange …) changes the water before corrosion control
- calculate doses, treated water quality and unit sizes for the method **you** choose
- get 'points to check' where your values differ from NV 212, NV 274 or the chemistry
- export a summary table, the calculation steps and a Deffeyes diagram for your report

**Builds on:** Part A (chemistry) and Part B (worked example) · **Time:** as long as your design needs
</div>

## C0 How to use this worksheet

<div style='border:1px solid rgba(229,57,53,0.6);border-left:5px solid #e53935;padding:6px 12px'>

**First: save your own copy.** *File → Save Notebook As…* and give it a name with your group number, e.g. `C_Group07_corrosion_control.ipynb`. Work only in your copy – then course updates can never overwrite your inputs.
</div>

**How it works.** Each section has an **input cell** (white code cell, visible) that you edit, followed by a **result cell** (code hidden) that calculates and shows the results. After changing an input cell, run it and all cells below it (*Run → Run Selected Cell and All Below*). The sections build on each other: C1 → C2 → C4 → C5 → C6 → C7.

**Every input is written as** `"key": (value, "unit", "source")`. The source is what you would cite in the report:

| What you write as source | Meaning | Shown as |
| --- | --- | --- |
| `"Client brief, Table 1 (lowest value)"`, `"NV 212/2015 Table 12.3"`, `"Lab 1 report, Fig. 3"` | a verifiable source | source |
| `"assumption: no data in the brief; typical for lake water (NV 212 §9.3.1: 1–5 mg Ca/L)"` | your assumption with its justification | assumption → goes into the assumption register |
| `"EXAMPLE (Part B) – replace"` | the example value that is in the cell now | ⚠ example value |
| `""` | nothing | ⚠ source missing |

For assumptions you can add a fourth element: what happens if the assumption is wrong, e.g. `("…", "°C", "assumption: …", "colder water → slower marble dissolution; check EBCT")`. It appears in the assumption register (report chapter 2).

**Points to check.** Below each result the notebook lists points where your values look unusual compared with NV 212/2015, NV 274/2022 or the chemistry. They are suggestions, not verdicts: a point can be fine if you justify it in the report. And an empty list does not mean the design is right.

**What the notebook does not do:** it does not choose the method, the design point or the design values, and it does not know whether your sources are correct. These decisions – and their justification – are your work, and they are what the report is assessed on. The rules of the student guide (section 6.2) apply: design values from AI output need a verifiable source.

**Units:** flows in `"L/s"`, `"m3/h"` or `"m3/d"`; conductivity in `"µS/cm"` or `"mS/m"` (1 mS/m = 10 µS/cm – check your brief!); alkalinity in `"mmol/L"`, `"meq/L"`, `"mg CaCO3/L"` or `"mg HCO3/L"`; calcium in `"mg/L"` or `"mmol/L"`; temperatures in °C.

In [ ]:
%matplotlib inline
import stab_views as sv          # look and helpers from Part A
import my_design_views as dv     # display functions of Part C
import my_design as md           # calculations and checks of Part C
sv.DARK = False                  # dark theme in JupyterLab? Set True, then Run All Cells again
sv.setup()

## C1 Design basis

Fill in the cell below for **your** plant (report chapter 2). Some hints:

* **Flows.** Q<sub>max</sub> is the design flow for the corrosion-control units (NV 212 ch. 2: maximum hour in the maximum day, unless there is equalisation; see B1). Q<sub>avg</sub> gives consumption and costs, Q<sub>min</sub> the smallest dosing rate. Think of gross vs. net production: backwash water is produced too.
* **Design value and range.** For pH, alkalinity and calcium give a design value **and** the lowest and highest value. The design value should represent the critical situation, not the average (student guide, criterion 1). The range is used for the dose range of the dosing equipment and for your sensitivity check.
* **Temperature.** If your brief gives none, make an assumption and justify it. `T_lab` is the temperature at which the pH was measured; if unknown, state your assumption.
* **Missing data** (e.g. no calcium in the brief): make an assumption, justify it, and say what happens if it is wrong.
* **Targets.** NV 274/2022: pH 8.0–8.5, alkalinity 0.6–1.0 mmol/L, Ca 15–25 mg/L (M08, slide 'Target water quality'). You may change them if your network requires it (M08, 'Targets per pipe material') – then justify it.

In [ ]:
basis = {
    # --- flows ("L/s", "m3/h" or "m3/d") ---
    "Q_max": (80, "L/s", "EXAMPLE (Part B) – replace"),
    "Q_avg": (50, "L/s", "EXAMPLE (Part B) – replace"),
    "Q_min": (15, "L/s", "EXAMPLE (Part B) – replace"),
    # --- temperatures (°C) ---
    "T_min": (2, "°C", "EXAMPLE (Part B) – replace"),
    "T_max": (12, "°C", "EXAMPLE (Part B) – replace"),
    "T_lab": (5, "°C", "EXAMPLE (Part B) – replace"),          # temperature at which the pH was measured
    # --- raw water: design value, lowest, highest ---
    "pH": (6.6, "-", "EXAMPLE (Part B) – replace"),
    "pH_min": (6.4, "-", "EXAMPLE (Part B) – replace"),
    "pH_max": (6.8, "-", "EXAMPLE (Part B) – replace"),
    "alk": (0.10, "mmol/L", "EXAMPLE (Part B) – replace"),
    "alk_min": (0.05, "mmol/L", "EXAMPLE (Part B) – replace"),
    "alk_max": (0.15, "mmol/L", "EXAMPLE (Part B) – replace"),
    "ca": (2.5, "mg/L", "EXAMPLE (Part B) – replace"),
    "ca_min": (2.5, "mg/L", "EXAMPLE (Part B) – replace"),
    "ca_max": (2.5, "mg/L", "EXAMPLE (Part B) – replace"),
    "ec": (35, "µS/cm", "EXAMPLE (Part B) – replace"),         # conductivity, "µS/cm" or "mS/m"
    # --- targets for the treated water ---
    "pH_t_lo": (8.0, "-", "NV 274/2022; M08 slide 'Target water quality'"),
    "pH_t_hi": (8.5, "-", "NV 274/2022; M08 slide 'Target water quality'"),
    "alk_t_lo": (0.6, "mmol/L", "NV 274/2022"),
    "alk_t_hi": (1.0, "mmol/L", "NV 274/2022"),
    "ca_t_lo": (15, "mg/L", "NV 274/2022"),
    "ca_t_hi": (25, "mg/L", "NV 274/2022"),
}

In [ ]:
dv.show_basis(basis)

## C2 Pre-treatment: the water that reaches corrosion control

Corrosion control is usually the **last** treatment step. The water it receives is not the raw water, but the water after coagulation, filtration, membranes, ion exchange or aeration – and these steps change exactly the quantities the corrosion-control design is based on. We have not discussed this much in class, so here is the overview:

| Process | Alkalinity | C<sub>T</sub> | Calcium | Why | Size of the effect |
| --- | --- | --- | --- | --- | --- |
| **Fe(III) coagulant** (e.g. FeCl₃, FeClSO₄) | ↓ | = | = | Fe³⁺ + 3 HCO₃⁻ → Fe(OH)₃ + 3 CO₂: the hydrolysis releases acid, which uses up bicarbonate; the CO₂ formed stays in the water (A7) | −0.054 mmol/L per mg Fe/L |
| **Al(III) coagulant** (e.g. Al₂(SO₄)₃, PAX) | ↓ | = | = | the same hydrolysis, 3 eq per mol Al | −0.111 mmol/L per mg Al/L; less for pre-polymerised products: × (1 − basicity) |
| **base before coagulation** | ↑ | = (soda: ↑) | lime: ↑ | sets the coagulation pH when the water's own alkalinity is too small | NaOH +0.025, Ca(OH)₂ +0.027 mmol/L per mg/L |
| **Fe(II)/Mn(II) oxidation** (groundwater) | ↓ | = | = | Fe²⁺ + ¼ O₂ + 2½ H₂O → Fe(OH)₃ + 2 H⁺ (Mn similar) | −0.036 mmol/L per mg Fe or Mn/L – small |
| **aeration / CO₂ stripping** | = | ↓ | = | only CO₂ leaves (A2, B7) | pH rises |
| **nanofiltration (NF)** | ↓ (partly) | ↓ (partly) | ↓↓ | divalent ions (Ca²⁺, Mg²⁺, SO₄²⁻) are rejected > 90 %, monovalent (HCO₃⁻) < 50 % (lecture *Membranes*, slide 'Typical performances'); dissolved CO₂ is a small uncharged molecule and passes | permeate: soft, low alkalinity, **same CO₂** → low pH, aggressive |
| **reverse osmosis (RO)** | ↓↓ | ↓↓ (except CO₂) | ↓↓ | > 98–99 % of all ions rejected (same slide); CO₂ passes | almost ion-free water + CO₂ → needs full remineralisation |
| **anion exchange** (Cl⁻ form, e.g. for NOM or PFAS) | ↓ at the start of a run | ↓ | = | HCO₃⁻ (and SO₄²⁻) are exchanged for Cl⁻ until the resin is loaded with them | can change over each run |
| **cation exchange** (softening, NV 212 §9.3.3) | = | = | ↓↓ | Ca²⁺ and Mg²⁺ exchanged for Na⁺ | – |
| GAC, UV, ozone | ≈ = | ≈ = | = | no significant effect on the carbonate system | – |

**Why it matters – three examples.**
1. A soft lake water with 0.15 mmol/L alkalinity and **4 mg Fe/L** coagulant loses 4 × 0.054 = 0.21 mmol/L – more than it has. The pH collapses unless a base is added before coagulation. At the same time about 0.2 mmol/L (9 mg/L) CO₂ is produced, which a marble filter can use: this is why the Molde process needs so little extra CO₂ (NV 212 §12.3.1).
2. **NF permeate** keeps its CO₂ but loses most of its calcium: it is more aggressive than the raw water, even if the raw water was fine. Blending with a bypass or remineralisation (CO₂ + marble uses the CO₂ that is already there) is then part of the design.
3. A **hard groundwater** may already meet the targets. Then corrosion control may only be needed *because* a pre-treatment step changes the water – or not at all.

**What the notebook calculates and what you provide.** For coagulation, base dosing and Fe/Mn oxidation the notebook applies the stoichiometry above to **your** dose. For membranes and ion exchange it cannot predict the performance: you give the rejection/removal fractions with a source (manufacturer data, pilot data, literature, the membrane lecture) and the share of the flow treated (the rest bypasses and is blended).

**Step formats** – copy the lines you need into the list below, in the order of your treatment train:

```python
{"step": "alkali", "chemical": "NaOH", "dose": 5.0, "source": "..."},          # "NaOH", "Ca(OH)2" or "Na2CO3"; mg/L of the chemical
{"step": "coagulant", "metal": "Fe", "dose": 4.0, "basicity": 0.0, "source": "..."},   # "Fe" or "Al"; mg metal/L
{"step": "fe_mn_oxidation", "fe": 0.6, "mn": 0.6, "source": "..."},            # mg/L oxidised and removed
{"step": "co2_dose", "dose": 5.0, "source": "..."},                            # mg CO2/L
{"step": "co2_strip_to", "residual": 15.0, "source": "..."},                   # mg CO2/L left
{"step": "membrane", "rej_ca": 0.9, "rej_alk": 0.5, "rej_co2": 0.0, "share": 1.0, "source": "..."},  # fractions 0–1
{"step": "ion_exchange", "rem_ca": 0.0, "rem_alk": 0.8, "share": 1.0, "source": "..."},           # fractions 0–1
```

In [ ]:
pretreatment = [
    # one dictionary per step, in the order of your treatment train (formats above), e.g.
    # {"step": "coagulant", "metal": "Fe", "dose": 4.0, "basicity": 0.0, "source": "Lab 1, optimum dose"},
]

In [ ]:
dv.show_pretreatment(pretreatment)

## C3 Choosing the method

The figure shows what single steps **could** do to the water entering corrosion control: marble alone, CO₂ + marble, stripping + marble, NaOH. It is an illustration, not a design and not a recommendation.

In [ ]:
dv.show_options()

**Your comparison** (double-click this cell to edit it; copy the table into your report chapter 3). The five methods this worksheet can calculate are listed; add waterglass or micronised marble if they are relevant for you (they cannot be calculated here – see A7 and NV 212 §12.3.2 / ch. 13).

| Criterion | CO₂ + marble filter | CO₂ stripping + marble filter | lime + CO₂ | coagulation + three-media filter (Molde) | coagulation + separate marble filter |
| --- | --- | --- | --- | --- | --- |
| adds Ca and alkalinity? | | | | | |
| fits the water entering corrosion control? | | | | | |
| fits the rest of the treatment train? | | | | | |
| operation, chemicals handling | | | | | |
| … your criteria | | | | | |
| **decision** | | | | | |

Sources for the comparison: NV 212/2015 ch. 12; M08 slides 'Three strategies in Norway' and 'Choosing a method'; Part B chapter B2.

## C4 Chemistry: method, design point and doses

Choose the method (exactly as written in the list) and the **design point** – the pH and alkalinity the treated water should have, and the temperature at which it should be met (B3). For *CO₂ stripping + marble filter* also give the CO₂ left after stripping (NV 212 §12.2.1: 12–15 mg/L; see B7). `naoh_trim` is an optional NaOH dose after the corrosion-control step (0 if none).

Methods: `"CO2 + marble filter"`, `"CO2 stripping + marble filter"`, `"lime + CO2"`, `"coagulation + three-media filter (Molde)"`, `"coagulation + separate marble filter"`.

The calculation uses the water from C2 at your design temperature: marble or lime from the alkalinity difference, CO₂ from the C<sub>T</sub> difference (B3); for stripping, the marble result is the equilibrium limit. Below the results you find the treated water at your temperature range, in houses and in hot-water systems, and the doses for your whole raw-water range.

In [ ]:
method = {
    "method": "CO2 + marble filter",
    "pH_target": (8.3, "-", "EXAMPLE (Part B) – replace"),
    "alk_target": (1.0, "mmol/L", "EXAMPLE (Part B) – replace"),
    "T_design": (2, "°C", "EXAMPLE (Part B) – replace"),
    "co2_after_stripping": (13, "mg/L", "EXAMPLE (Part B) – replace"),   # only used for "CO2 stripping + marble filter"
    "naoh_trim": (0, "mg/L", "EXAMPLE (Part B) – replace"),
}

In [ ]:
dv.show_chemistry(method)

**Trying other design points.** The sliders below start from your method cell. Use them to explore; then write the values you decide on into the method cell – with your reasoning as the source – and run it again. Slider positions are not saved.

In [ ]:
dv.explore_panel()

## C5 Units: dimensioning

Fill in the design values for the units your method needs; the other sections are ignored. Each value needs **your** source – the example values point to where they come from in Part B, but the choice within a range is yours (e.g. EBCT 20 or 30 min?) and must be justified. The units are sized for Q<sub>max</sub> with one unit out of service (B4).

| Method | Sections used |
| --- | --- |
| CO₂ + marble filter; coagulation + separate marble filter; Molde | `marble_filter`, `co2` |
| CO₂ stripping + marble filter | `stripping_tower`, `marble_filter` |
| lime + CO₂ | `lime_saturator`, `co2` |
| any method with `naoh_trim` > 0 | `naoh` |

In [ ]:
units = {
    "marble_filter": {
        "n": (4, "-", "EXAMPLE (NV 212 Table 4.16: at least 2, preferably 4) – replace"),
        "ebct_min": (20, "min", "EXAMPLE (NV 212 Table 12.3: ≥ 20 min; §12.2.2.4: 20–30 min) – replace"),
        "v": (8, "m/h", "EXAMPLE (NV 212 §12.2.2.4: downflow 8–10 m/h; Table 12.3: Molde ≤ 7.5 m/h) – replace"),
        "depth_min": (2.5, "m", "EXAMPLE (NV 212 Table 12.3: ≥ 2.5 m) – replace"),
        "reserve": (0.3, "m", "EXAMPLE (assumption in Part B4) – replace"),
        "v_bw": (60, "m/h", "EXAMPLE (NV 212 Table 12.3: ≥ 60 m/h; Table 4.19: 50–65 m/h) – replace"),
        "t_bw": (10, "min", "EXAMPLE (NV 212 Table 4.19: 8–12 min) – replace"),
        "bw_per_week": (1, "1/week", "EXAMPLE (assumption in Part B4) – replace"),
        "rho_bulk": (1.5, "t/m3", "EXAMPLE (Visnes Kalk data sheet: 1.45–1.55 kg/dm³) – replace"),
        "purity": (0.98, "-", "EXAMPLE (Visnes Kalk data sheet: ≥ 98 % CaCO3) – replace"),
        "loss": (0.05, "-", "EXAMPLE (assumption in Part B5; M08 'Molde process': 10–15 % in three-media filters) – replace"),
    },
    "stripping_tower": {
        "efficiency": (0.90, "-", "EXAMPLE (NV 212 §8.2.3.4: 90–95 %) – replace"),
        "vf": (60, "m/h", "EXAMPLE (NV 212 §8.2.4.4: 40–100 m3/(m2·h)) – replace"),
        "depth": (4, "m", "EXAMPLE (NV 212 §8.2.4.4: 3–5 m) – replace"),
        "air_water": (10, "-", "EXAMPLE (NV 212 Table 8.2: 5–100 for towers) – replace"),
        "n": (2, "-", "EXAMPLE (Part B7) – replace"),
    },
    "lime_saturator": {
        "conc": (1.5, "g/L", "EXAMPLE (g Ca(OH)2/L; solubility ≈ 1.7 g/L at 20 °C, see A7) – replace"),
        "surface_load": (2, "m/h", "EXAMPLE (NV 212 Table 12.2: 2 m/h without polymer/lamellae) – replace"),
        "contact_time": (50, "min", "EXAMPLE (NV 212 Table 12.2: 40–60 min) – replace"),
        "n": (2, "-", "EXAMPLE – replace"),
        "purity": (0.90, "-", "EXAMPLE (Ca(OH)2 content of the product; supplier data sheet) – replace"),
    },
    "co2": {
        "utilisation": (0.90, "-", "EXAMPLE (assumption in Part B5) – replace"),
        "storage_days": (30, "d", "EXAMPLE (assumption in Part B5) – replace"),
        "rho_liquid": (1.03, "t/m3", "EXAMPLE (NIST: liquid CO2 at −20 °C) – replace"),
    },
    "naoh": {
        "conc": (25, "%", "EXAMPLE (% w/w; supplier data sheet) – replace"),
        "density": (1.27, "kg/L", "EXAMPLE (25 % NaOH; supplier data sheet) – replace"),
        "storage_days": (30, "d", "EXAMPLE – replace"),
    },
}

In [ ]:
dv.show_units(units)

## C6 Chemical costs (optional – estimates)

Give a price in NOK/kg for each chemical your design uses, with a source (supplier offer, tender, literature) – or delete the line. The result is an **estimate** of the running costs for chemicals only; it is meant for comparing alternatives (report chapter 7). Part B5 shows how such prices can be justified. Leave the dictionary empty (`prices = {}`) to skip this section.

In [ ]:
prices = {
    "co2": (3.0, "NOK/kg", "EXAMPLE (Part B) – replace (see B5: ESTIMATE)"),
    "marble": (2.0, "NOK/kg", "EXAMPLE (Part B) – replace (see B5: ESTIMATE)"),
    # "lime": (..., "NOK/kg", "..."),
    # "naoh": (..., "NOK/kg", "... (price per kg 100 % NaOH)"),
}

In [ ]:
dv.show_costs(prices)

## C7 Summary and export

The table collects every input and result with its unit, source and type. Before you use it: no line should say *source missing* or *example value*. The buttons save

* the table as **CSV** (opens in Excel – e.g. for your calculation appendix),
* the table plus all calculation steps as **HTML** (open in a browser, copy into Word),
* the Deffeyes diagram of your treatment path as **PNG** (white background, for the report).

The files are saved in the folder of this notebook; download them from the file browser on the left (right-click → Download).

In [ ]:
dv.show_summary()

## C8 What the report needs from this worksheet

The student guide (section 4) describes the report. For the corrosion-control step this means:

| Report chapter (student guide) | From this worksheet | Do not forget |
| --- | --- | --- |
| 2 Design basis | C1: flows, raw water with range, targets with source; assumption register | design values for the critical situation, not averages; consequences of assumptions |
| 3 Options and selection | C3: your comparison table | why the other methods were ruled out; weaknesses of your choice |
| 4 Treatment scheme | position of corrosion control in the flow diagram; dosing points (CO₂, NaOH …); C2 shows the link to the steps before | link to disinfection (B6): pH and chlorine C·t, turbidity before UV |
| 5 Dimensioning | C4 doses and treated water; C5 units with design criteria and sources; summary table | Q<sub>max</sub> with one unit out, Q<sub>min</sub>; equations with symbols and units (guide 5.2) |
| 6 Residuals, operation, monitoring | backwash water (C5), refill, dosing range | outlet pH as key indicator (B6) |
| 7 Cost, risk and uncertainty | C6 cost estimate; the raw-water range table in C4 | sensitivity: which assumption changes the design most? |

Every number in the report must be traceable to your calculation appendix (guide 5.2): the CSV/HTML export of C7 is a good start, but say in the report that the calculations were made with this worksheet and which equations it uses (Part B3–B5 explains them).

---
<div style='text-align:right;font-size:0.9em'><a href='B8_Calculator_and_summary.ipynb'>← B8 Design calculator and summary</a> &nbsp;·&nbsp; <a href='00_Start_here.ipynb'>Contents</a></div>